In [1]:
!pip install Unidecode
!pip install rapidfuzz


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 235.8/235.8 kB 3.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.2/3.2 MB 34.4 MB/s eta 0:00:00


In [2]:
# Librerias
import pandas as pd
from google.colab import drive
import unicodedata
import unidecode
import re
from rapidfuzz import process, fuzz
import numpy as np

### 0. Carga de datos

In [4]:
drive.mount('/content/drive')

MessageError: Error: credential propagation was unsuccessful

In [ ]:
path = "/content/drive/MyDrive/Visualizacion & Storytelling/Trabajo_final_storytelling/Entrega2/Insumos/SALARY_NAMES.xlsx"
df_dirty = pd.read_excel(path)



## Renombrar las Columnas:
Debido a que tienen nombres demasiado extensos, es encesario renombrar las variables con palabras claves, para evitar confusiones y eliminar ruido en los nombres

In [ ]:
rename_map = {
    # Fecha
    "Timestamp": "Timestamp",
    # Edad
    "How old are you?": "Age",
    # Industria / trabajo
    "What industry do you work in?": "Industry",
    "Job title": "Job_title",
    "If your job title needs additional context, please clarify here:": "Job_details",
    # Salarios
    "What is your annual salary? (You'll indicate the currency in a later question. If you are part-time or hourly, please enter an annualized equivalent -- what you would earn if you worked the job 40 hours a week, 52 weeks a year.)": "Annual_salary",
    "How much additional monetary compensation do you get, if any (for example, bonuses or overtime in an average year)? Please only include monetary compensation here, not the value of benefits.": "Aditional_salary",
    # Moneda
    "Please indicate the currency": "Currency",
    'If "Other," please indicate the currency here: ': "Other_currency",
    # Contexto ingresos
    "If your income needs additional context, please provide it here:": "Income_details",
    # Ubicación
    "What country do you work in?": "Country",
    "If you're in the U.S., what state do you work in?": "State",
    "What city do you work in?": "City",
    # Experiencia
    "How many years of professional work experience do you have overall?": "Years_experience_overall",
    "How many years of professional work experience do you have in your field?": "Years_experience_field",
    # Educación
    "What is your highest level of education completed?": "Max_education_level",
    # Demografía
    "What is your gender?": "Gender",
    "What is your race? (Choose all that apply.)": "Race",
}

df = df_dirty.rename(columns=rename_map)

In [ ]:
df.columns

In [ ]:
df['Timestamp'].min()

In [ ]:
df['Timestamp'].max()

In [ ]:
df.shape

In [ ]:
df.nunique()

In [ ]:
df.describe()

## Revisar valores duplicados, nulos y porcentaje de nulos
Para evitar datos que no son necesarios, y datos que van a generar ruido, se realiza una tabla especificando cada una de las columnas con valores unicos, nulos y porcentaje de los ultimos

In [ ]:
perfil = pd.DataFrame({
    "valores_unicos": df.nunique(),
    "nulos": df.isna().sum(),
    "porcentaje_nulos": (df.isna().sum() / len(df)) * 100
})

perfil.sort_values("valores_unicos", ascending=False)

### **Eliminar del DF columnas no necesarias**
#### **TimeStamp**: Variable relevante para informar el periodo de los datos.
#### **info_adicionales_trabajo**:  es una variable que no es parametrizable facilmente, ademas para la información que va a ser usada en el dashboard no generan impacto, sería necesario ver los datos de cada una de las personas.
#### **variables con valores nulos** que no tengan impacto en el % mayor a 1% entre estas variables estan: nombre_trabajo, ciudad, industria, etnia, nivel_educacion, genero.

In [ ]:
# Conservar timestamp para saber el inicio y fin de la base de datos, castear a fecha
df["Timestamp"] = pd.to_datetime(df["Timestamp"])
df["Timestamp"] = df["Timestamp"].dt.date

df = df.drop(columns=["Job_details"])

cols_valores_nulos = [
    "Job_title",
    "Industry",
    "Race",
    "Gender",
    "Max_education_level"
]


print("Registros iniciales:", df.shape[0])
df = df.dropna(subset=cols_valores_nulos)
print("Registros luego de limpieza:", df.shape[0])

In [ ]:
df[cols_valores_nulos].isna().sum()

#### Validaciony limpieza de la columna Country

In [ ]:
# Normalizar

In [ ]:
df['Country'].unique()

In [ ]:
# Capa 1, limpieza de comas, espacios, etc.

df['country_clean'] = (
    df['Country']
    .astype(str)
    .apply(unidecode.unidecode)
    .str.lower()
    .str.strip()
)

# quitar símbolos raros
df['country_clean'] = df['country_clean'].apply(lambda x: re.sub(r'[^a-z\s]', '', x))

# quitar dobles espacios
df['country_clean'] = df['country_clean'].str.replace(r'\s+', ' ', regex=True)

In [ ]:
df['country_clean'].unique()

In [ ]:
def norm_country_raw(x):
    if pd.isna(x):
        return np.nan
    x = str(x).strip()
    x = re.sub(r"\s+", " ", x)
    if x == "":
        return np.nan
    return x

def norm_key(x):
    if pd.isna(x):
        return np.nan
    x = str(x).strip().lower()
    x = "".join(c for c in unicodedata.normalize("NFD", x) if unicodedata.category(c) != "Mn")
    x = re.sub(r"\s+", " ", x)
    return x

df["country_clean"] = df["country_clean"].apply(norm_country_raw)
df["country_clean"] = df["country_clean"].apply(norm_key)

Eliminar duplicados

In [ ]:
bad_patterns = [
    r"we don't get raises",
    r"i work for",
    r"i am located",
    r"worldwide",
    r"global",
    r"remote",
    r"currently",
    r"policy",
    r"contracts",
    r"deducted",
    r"commission",
    r"quarterly",
    r"bonus",
    r"salary",
    r"benefits",
    r"country withheld",
    r"ss$|dbfemf|loutreland|uxz",   # basura evidente
    r"^\$",
    r"usd$",                       # 'USD' no es país
    r"^y$",
    r"^na$",
    r"^n/a",
]

bad_regex = re.compile("|".join(bad_patterns), re.IGNORECASE)

df["pais_no_pais"] = (
    df["country_clean"].astype(str).str.len().gt(35) |
    df["country_clean"].astype(str).str.contains(bad_regex, na=False) |
    df["country_clean"].astype(str).str.contains(r"\d", na=False)  # números en país
)

### Mapeo de los paises

In [ ]:
# Reemplazar valores directamente

dic_paises = {
    # 🇺🇸
    'us': 'United States',
    'usa': 'United States',
    'united states': 'United States',
    'united states of america': 'United States',
    'america': 'United States',
    'the us': 'United States',

    'unted states': 'United States',
    'united statesp': 'United States',
    'united stattes': 'United States',
    'united statea': 'United States',
    'united statees': 'United States',
    'united sates of america': 'United States',
    'united states of america': 'United States',
    'Usaa': 'United States',
    'u s': 'United States',
    'uganda': 'United States',
    'uniited states': 'United States',
    'unite states': 'United States',
    'united sates': 'United States',
    'united stares': 'United States',
    'united state': 'United States',
    'united state of america': 'United States',
    'united stated': 'United States',
    'united stateds': 'United States',
    'united states i work from home and my clients are all over the uscanadapr': 'United States',
    'united states is america': 'United States',
    'united states of american': 'United States',
    'united states of americas': 'United States',
    'united states puerto rico': 'United States',
    'united statew': 'United States',
    'united statss': 'United States',
    'united statues': 'United States',
    'united status': 'United States',
    'united statws': 'United States',
    'united sttes': 'United States',
    'united y': 'United States',
    'unitedstates': 'United States',
    'uniteed states': 'United States',
    'unitef stated': 'United States',
    'uniter statez': 'United States',
    'unites states': 'United States',
    'unitied states': 'United States',
    'uniyed states': 'United States',
    'uniyes states': 'United States',
    'untied states': 'United States',
    'us govt employee overseas country withheld': 'United States',
    'us of a': 'United States',
    'usa but for foreign govt': 'United States',
    'usa company is based in a us territory i work remote': 'United States',
    'usa tomorrow': 'United States',
    'usa virgin islands': 'United States',
    'usa work from home': 'United States',
    'the united states': 'United States',
    'for the united states government but posted overseas': 'United States',
    'u.s.': 'United States',
    'u.s': 'United States',
    'u.s.a': 'United States',
    'u.s.a.': 'United States',
    '🇺🇸': 'United States',
    'united states ': 'United States',
    'united  states': 'United States',


    # 🇬🇧
    'u.k.': 'United Kingdom',
    'u.k': 'United Kingdom',
    'uk': 'United Kingdom',
    'united kingdom': 'United Kingdom',
    'england': 'United Kingdom',
    'scotland': 'United Kingdom',
    'wales': 'United Kingdom',
    'great britain': 'United Kingdom',
    'britain': 'United Kingdom',
    'uk but for globally fully remote company' : 'United Kingdom',
    'uk england' : 'United Kingdom',
    'uk for us company' : 'United Kingdom',
    'uk northern england' : 'United Kingdom',
    'uk northern ireland' : 'United Kingdom',
    'uk remote' : 'United Kingdom',
    'united kindom' : 'United Kingdom',
    'united kingdom england' : 'United Kingdom',
    'united kingdomk' : 'United Kingdom',
    'unites kingdom' : 'United Kingdom',
    'wales uk': 'United Kingdom',
    'wales united kingdom': 'United Kingdom',
    'england gb': 'United Kingdom',
    'england uk': 'United Kingdom',
    'england united kingdom': 'United Kingdom',
    'englanduk': 'United Kingdom',
    'englang': 'United Kingdom',
    'london': 'United Kingdom',
    'northern ireland united kingdom': 'United Kingdom',
    'scotland uk': 'United Kingdom',
    'scotland united kingdom': 'United Kingdom',
    'isle of man': 'United Kingdom',

    # Arg
    'argentina': 'Argentina',
    'argentina but my org is in thailand': 'Argentina',

    # 🇨🇦
    'canada': 'Canada',
    'can': 'Canada',
    'canada but for us': 'Canada',
    'canad': 'Canada',
    'canada and usa': 'Canada',
    'canada ottawa ontario': 'Canada',
    'canadw': 'Canada',
    'canda': 'Canada',
    'csnada': 'Canada',

    # 🇦🇺
    'australia': 'Australia',
    'australi': 'Australia',
    'australian': 'Australia',

    # 🇳🇱
    'netherlands': 'Netherlands',
    'the netherlands': 'Netherlands',
    'nederland': 'Netherlands',

    # 🇩🇪
    'germany': 'Germany',
    'company in germany i work from pakistan': 'Germany',

    # Sui
    'switzerland': 'Switzerland',

    # 🇫🇷
    'france': 'France',

    # 🇪🇸
    'spain': 'Spain',

    # 🇮🇪
    'ireland': 'Ireland',
    'northern ireland': 'Ireland',

    # 🇮🇳
    'india': 'India',

    #Jap

    'japan': 'Japan',
    'japan us gov position': 'Japan',

    # Chi
    'hong kong': 'China',
    'hong kongkong': 'China',
    'hong konh': 'China',
    'china': 'China',
    'mainland china': 'China',

    # Ni
    'nigeria': 'Nigeria',
    'nigeria uk': 'Nigeria',

    # Nw Ze
    'new zealand': 'New Zealand',
    'new zealand aotearoa': 'New Zealand',
    'aotearoa new zealand': 'New Zealand',
    'from new zealand but on projects across apac': 'New Zealand',
    'nz': 'New Zealand',
    'aotearoa new zealand': 'New Zealand',

    #Br
    'brasil': 'Brazil',
    'brazil': 'Brazil',

    #Aus

    'austria': 'Austria',
    'austria but i work remotely for a dutchbritish company':'Austria',

    # Romania
    'from romania but for an us based company': 'Romania',
    'romania': 'Romania',

    # UAB

    'united arab emirates': 'United Arab Emirates',
    'uae': 'United Arab Emirates',

    # OTR

    'burma': 'Myanmar',
    'czechia': 'Czech Republic',
    'luxemburg': 'Luxembourg',
    'italia': 'Italy',
    'cote divoire': "Côte d'Ivoire",
    'ceska republika': 'Czech Republic',
    'danmark': 'Denmark',

    # idioma / abreviación
    'nl': 'Netherlands',
    'ua': 'Ukraine',

    # regiones dentro de US
    'california': 'United States',
    'virginia': 'United States',
    'san francisco': 'United States',
    'hartford': 'United States',

    # territorios / decisiones de negocio
    'puerto rico': 'United States',

    # variaciones
    'italy south': 'Italy',
    'remote philippines': 'Philippines',

    # nombres políticos / regionales
    'catalonia': 'Spain',

    # variaciones comunes
    'the bahamas': 'Bahamas',

    # ambiguos pero normalmente
    'jersey channel islands': 'United Kingdom',

    ############################
    # Adicionales
    'canadá': 'Canada',

    # CZ (nombres en lista 2)
    'czech republic': 'Czechia',
    'česká republika': 'Czechia',

    # Puerto Rico / Hong Kong / Taiwan (tal cual lista 2; OJO: en tu lista 1 ya tienes PR -> US, aquí NO lo toco)
    'hong kong': 'Hong Kong',
    'taiwan': 'Taiwan',

    # Europa (presentes en lista 2 y ausentes en lista 1)
    'finland': 'Finland',
    'sweden': 'Sweden',
    'norway': 'Norway',
    'portugal': 'Portugal',
    'poland': 'Poland',
    'belgium': 'Belgium',
    'croatia': 'Croatia',
    'cyprus': 'Cyprus',
    'estonia': 'Estonia',
    'hungary': 'Hungary',
    'latvia': 'Latvia',
    'lithuania': 'Lithuania',
    'malta': 'Malta',
    'serbia': 'Serbia',
    'slovakia': 'Slovakia',
    'slovenia': 'Slovenia',
    'ukraine': 'Ukraine',
    'russia': 'Russia',

    # LatAm (presentes en lista 2 y ausentes en lista 1)
    'chile': 'Chile',
    'colombia': 'Colombia',
    'costa rica': 'Costa Rica',
    'ecuador': 'Ecuador',
    'jamaica': 'Jamaica',
    'mexico': 'Mexico',
    'panama': 'Panama',
    'panamá': 'Panama',
    'trinidad and tobago': 'Trinidad and Tobago',
    'uruguay': 'Uruguay',

    # Asia / ME (presentes en lista 2 y ausentes en lista 1)
    'indonesia': 'Indonesia',
    'israel': 'Israel',
    'jordan': 'Jordan',
    'kuwait': 'Kuwait',
    'malaysia': 'Malaysia',
    'pakistan': 'Pakistan',
    'qatar': 'Qatar',
    'saudi arabia': 'Saudi Arabia',
    'singapore': 'Singapore',
    'south korea': 'South Korea',
    'sri lanka': 'Sri Lanka',
    'thailand': 'Thailand',
    'vietnam': 'Vietnam',

    # África (presentes en lista 2 y ausentes en lista 1)
    'ghana': 'Ghana',
    'kenya': 'Kenya',
    'morocco': 'Morocco',
    'south africa': 'South Africa',
    'tanzania': 'Tanzania',
    'uganda': 'Uganda',
    'zambia': 'Zambia',
    'zimbabwe': 'Zimbabwe'

}




In [ ]:
df['country_std'] = df['country_clean'].replace(dic_paises)
df["pais_invalido"] = df["country_clean"].isna() | df["pais_no_pais"]
print("Registros inválidos (se eliminarán):", df["pais_invalido"].sum())

df = df[~df["pais_invalido"]].copy()


In [ ]:
df.loc[df["country_std"].isna(), "country_clean"].value_counts().head(50)

In [ ]:
print(" Numero de paises totales: " ,df["country_std"].nunique())
sorted(df['country_std'].unique())

In [ ]:
# Capa final 1, capitalizar y hacer

df['country_std'] = df['country_std'].str.lower().str.strip()
df['country_std'] = df['country_std'].str.title()

In [ ]:
sorted(df['country_std'].unique())

In [ ]:
# Capa Final 2, Unknown

no_paises = [
'Africa','Europe','Global','International','Remote','Na',
'Policy','Contracts','Currently Finance','Dbfemf','Ff','Is','Isa',
'Loutreland','Ibdia','Ss','Y',
'Usaa','Usab','Usat','Usd','Uss','Uxz',
'Year Is Deducted For Benefits',
'Bonus Based On Meeting Yearly Goals Set W My Supervisor',
'We Dont Get Raises We Get Quarterly Bonuses But They Periodically Asses Income In The Area You Work So I Got A Raise Because A Rd Party Assessment Showed I Was Paid Too Little For The Area We Were Located',
'Worldwide Based In Us But Short Term Trips Aroudn The World',
'I Am Located In Canada But I Work For A Company In The Us',
'I Earn Commission On Sales If I Meet Quota Im Guaranteed Another K Min Last Year I Earned An Additional K Its Not Uncommon For People In My Space To Earn K After Commission',
'I Was Brought In On This Salary To Help With The Ehr And Very Quickly Was Promoted To Current Position But Compensation Was Not Altered',
'I Work For A Uaebased Organization Though I Am Personally In The Us',
'I Work For An Us Based Company But Im From Argentina','Na Remote From Wherever I Want'
]

paises_validos = [p for p in df['country_std'].unique() if p not in no_paises and p != '']

paises_validos

In [ ]:
df.loc[~df['country_std'].isin(paises_validos), 'country_std'] = 'Other / Unknown'

In [ ]:
print(" Numero de paises totales: " ,df["country_std"].nunique())
sorted(df['country_std'].unique())

### Validar y limpiar la ciudad


### Imputar faltantes con la capital


In [ ]:
capital_map = {
    "United States": "Washington, D.C.",
    "Canada": "Ottawa",
    "Mexico": "Mexico City",
    "Puerto Rico": "San Juan",
    "Trinidad and Tobago": "Port of Spain",
    "Jamaica": "Kingston",
    "Costa Rica": "San José",
    "Panama": "Panama City",
    "Argentina": "Buenos Aires",
    "Brazil": "Brasília",
    "Chile": "Santiago",
    "Colombia": "Bogotá",
    "Ecuador": "Quito",
    "Uruguay": "Montevideo",
    "United Kingdom": "London",
    "Netherlands": "Amsterdam",
    "Finland": "Helsinki",
    "France": "Paris",
    "Germany": "Berlin",
    "Ireland": "Dublin",
    "Denmark": "Copenhagen",
    "Spain": "Madrid",
    "Switzerland": "Bern",
    "Belgium": "Brussels",
    "Sweden": "Stockholm",
    "Norway": "Oslo",
    "Austria": "Vienna",
    "Hungary": "Budapest",
    "Luxembourg": "Luxembourg",
    "Czechia": "Prague",
    "Latvia": "Riga",
    "Romania": "Bucharest",
    "Serbia": "Belgrade",
    "Poland": "Warsaw",
    "Lithuania": "Vilnius",
    "Italy": "Rome",
    "Slovenia": "Ljubljana",
    "Slovakia": "Bratislava",
    "Portugal": "Lisbon",
    "Estonia": "Tallinn",
    "Croatia": "Zagreb",
    "Ukraine": "Kyiv",
    "Cyprus": "Nicosia",
    "Malta": "Valletta",
    "Russia": "Moscow",
    "India": "New Delhi",
    "Malaysia": "Kuala Lumpur",
    "Kuwait": "Kuwait City",
    "Sri Lanka": "Sri Jayawardenepura Kotte",
    "Philippines": "Manila",
    "China": "Beijing",
    "Japan": "Tokyo",
    "Taiwan": "Taipei",
    "Vietnam": "Hanoi",
    "Singapore": "Singapore",
    "South Korea": "Seoul",
    "Thailand": "Bangkok",
    "Israel": "Jerusalem",
    "Indonesia": "Jakarta",
    "Pakistan": "Islamabad",
    "Jordan": "Amman",
    "United Arab Emirates": "Abu Dhabi",
    "Qatar": "Doha",
    "Saudi Arabia": "Riyadh",
    "South Africa": "Pretoria",
    "Nigeria": "Abuja",
    "Uganda": "Kampala",
    "Morocco": "Rabat",
    "Zimbabwe": "Harare",
    "Ghana": "Accra",
    "Kenya": "Nairobi",
    "Tanzania": "Dodoma",
    "Zambia": "Lusaka",
    "Australia": "Canberra",
    "New Zealand": "Wellington",
    "Hong Kong": "Hong Kong",
}

In [ ]:
# Contar Nulos

df["City"].isna().sum()
# df["City"].notna().sum()


In [ ]:
mask_ciudad_falta = df["City"].isna() | (df["City"].astype(str).str.strip() == "")

df["City"] = df["City"].where(~mask_ciudad_falta, df["country_std"].map(capital_map))

# Flag para documentar qué filas fueron imputadas

df["ciudad_imputada_capital"] = mask_ciudad_falta


In [ ]:
df["City"].isna().sum()

In [ ]:
df_null = df[df["City"].isna()]
df_null

In [ ]:
df['City'].unique()

In [ ]:
# Limpiar inicial
# Capa 1, limpieza de comas, espacios, etc.
df['city_clean'] = (
    df['City']
    .astype(str)
    .apply(unidecode.unidecode)
    .str.lower()
    .str.strip()
)

# quitar símbolos raros
df['city_clean'] = df['city_clean'].apply(lambda x: re.sub(r'[^a-z\s]', '', x))
df['city_clean'] = df['city_clean'].str.replace(r'\b[a-z]{2}$', '', regex=True).str.strip()

# quitar dobles espacios
df['city_clean'] = df['city_clean'].str.replace(r'\s+', ' ', regex=True)

# Otros ajustes

df['city_clean'] = df['city_clean'].str.replace(r'\bst\b', 'saint', regex=True)
df['city_clean'] = df['city_clean'].str.title()


In [ ]:
# Ajustes Manuales

city_manual = {
    # New York
    "nyc": "New York",
    "Nyc": "New York",
    "nyc metro": "New York",
    "new york city": "New York",
    "New York City": "New York",
    "New York Citymanhattan": "New York",
    "New York Citybrooklyn": "New York",
    "New York City Suburbs": "New York",
    "New York City The Bronx": "New York",
    " New York": "New York",
    "Austin New York": "New York",
    "Greater New York Metro Area": "New York",
    "New York Buty": "New York",
    "New Yorkncity": "New York",
    "Northern New York": "New York",
    "Upstate New York": "New York",
    " New York": "New York",
    "New York Metro": "New York",
    "New York City Remote": "New York",


    # Chicago
    "chicago suburbs": "Chicago",
    "chicagoland": "Chicago",
    "Chicago Remote": "Chicago",
    "Remote Chicago": "Chicago",
    "Suburb Of Chicago": "Chicago",
    "Suburban Chicago": "Chicago",
    "Greater Chicago": "Chicago",
    "Suburb Of Chicago": "Chicago",
    "Chicago Suburbs": "Chicago",
    "Chicago But The Company Is Remote": "Chicago",
    "Chicago Suburbs": "Chicago",
    "Chicago Area": "Chicago",
    "Chicago Suburbs": "Chicago",
    "Chicago Suburbs": "Chicago",
    "Chicago Suburbs": "Chicago",
    "Schaumburg Chicago Metro Area": "Chicago",
    "Chicago Suburbs": "Chicago",
    "Chicagoland Area": "Chicago",
    "Chicago Suburbs": "Chicago",
    "Chicago Suburbs": "Chicago",
    "Chicago Suburbs": "Chicago",
    "Chicagoland": "Chicago",
    "Chicago Area": "Chicago",
    "Chicago Suburbs": "Chicago",
    "Chicago Area": "Chicago",
    "Chicago Suburbs": "Chicago",
    "Affluent Chicago Suburb": "Chicago",
    "Suburban Chicago": "Chicago",
    "Suburban Chicago": "Chicago",
    "Chicago Suburbs": "Chicago",
    "Chicago Metro": "Chicago",
    "Chicago Suburbs": "Chicago",
    "Chicago But Distributed Workplace": "Chicago",
    "Home Plus Travel To Customers Near Chicago": "Chicago",
    "Chicago Suburbs": "Chicago",
    "Chicago Suburbs": "Chicago",
    "Chicagoland": "Chicago",
    "Greater Chicago Area": "Chicago",
    "Greater Chicagoland": "Chicago",
    "Chicago Suburbs": "Chicago",
    "Chicagodeerfield Two Offices": "Chicago",
    "Greater Chicago Area": "Chicago",
    "Chicago Suburbs": "Chicago",
    "Chicago Area": "Chicago",
    "Chicago Area": "Chicago",
    "Chicagoland": "Chicago",
    "Greater Chicagoland": "Chicago",
    "Chicago Remote Position Though": "Chicago",
    "Chicago Suburbs": "Chicago",
    "Near Chicago": "Chicago",
    "Chicago Suburbs": "Chicago",
    "From Home For A Chicago Based Company": "Chicago",
    "Chicago Suburbs": "Chicago",
    "Chicago Suburbs": "Chicago",
    "Chicago Metro": "Chicago",
    "Hoffman Estates Suburb Of Chicago": "Chicago",
    "Naperville Chicago Area": "Chicago",
    "Northwest Chicago Suburbs": "Chicago",
    "Chicagoland": "Chicago",
    "Chicago Suburbs": "Chicago",
    "Chicagoevanston": "Chicago",
    "Chicago Suburbs": "Chicago",
    "Chicago Ft Remote": "Chicago",

     # Boston
    "Greater Boston Area": "Boston",
    "Metro Boston": "Boston",
    "Fully Remote Greater Boston": "Boston",
    "Eastern Ma Not Boston": "Boston",
    "Boston Suburbs": "Boston",
    "Boston Area": "Boston",
    "Boston Massachusetts": "Boston",
    "Boston Area": "Boston",
    "Boston Suburbs": "Boston",
    "Greater Boston Area": "Boston",
    "Bostonish": "Boston",
    "North Of Boston": "Boston",
    "A Suburb Of Boston": "Boston",
    "Greater Boston Area": "Boston",
    "Boston Area": "Boston",
    "Metrowest Boston": "Boston",
    "Boston Area": "Boston",
    "Boston Greater Boston Metro Area": "Boston",
    "Greater Boston Area": "Boston",
    "Outside Of Boston": "Boston",
    "Metro Boston": "Boston",
    "Watertown Greater Boston Area": "Boston",
    "Waltham Suburb Of Boston": "Boston",
    "Boston Area": "Boston",
    "Boston Area": "Boston",
    "Lynnfield Boston Metro Area": "Boston",
    "Boston Area": "Boston",
    "Boston Metro Area": "Boston",
    "Bostoncambridge": "Boston",
    "Denverboston": "Boston",
    "Cambridgeboston": "Boston",
    "Near Boston": "Boston",
    "Boston Area": "Boston",
    "Greater Boston Area": "Boston",
    "Boston Area": "Boston",
    "Boston Metro": "Boston",
    "Boston Suburb": "Boston",
    "Boston Suburb": "Boston",
    "Boston Metro": "Boston",
    "Boston Area": "Boston",
    "Greater Boston": "Boston",
    "North Of Boston": "Boston",
    "Greater Boston": "Boston",
    "Boston Metro Area": "Boston",
    "Remote Live Near Boston": "Boston",
    "Salem Remote For Boston Mass": "Boston",
    "Greater Boston": "Boston",
    "Charlestown Boston": "Boston",
    "Boston Area": "Boston",
    "Boston Area": "Boston",
    "Cambridge Boston Metro Area": "Boston",
    "Boston Area": "Boston",
    "Outside Boston": "Boston",
    "Metrowest Boston Area": "Boston",
    "Boston Somerville Specifically": "Boston",
    "Boisebut Remote Company Located In Boston": "Boston",

    # Saint Louis (por si el catálogo trae Saint-Louis)
    "saint louis": "Saint-Louis",
    "Saint Louis": "Saint-Louis",

    # Ottawa (si tu catálogo usa Ottawa tal cual)
    "ottawa area": "Ottawa",

    # Prefer not / basura -> Unknown
    "prefer not to identify": "Other / Unknown",
    "variety": "Other / Unknown",
    "rural area": "Other / Unknown",
    "small town": "Other / Unknown",
    "northern": "Other / Unknown",
    "capital region": "Other / Unknown",

    # Ambiguos / no ciudad clara
    "montgomery county": "Other / Unknown",
    "lacey township": "Other / Unknown"
}

df["city_clean"] = df["city_clean"].replace(city_manual)

### Cargar ciudades para ajustar con Libreria Fuzzy

In [ ]:
# Cargar base de ciudades (fuente: Kaggle)
# Objetivo: Comparar ciudades oficiales con la libreria Fuzzy
pathCty = "/content/drive/MyDrive/Visualizacion & Storytelling/Trabajo_final_storytelling/Entrega2/Insumos/worldcities.csv"
cities = pd.read_csv(pathCty)



In [ ]:
cities.head(3)

In [ ]:
catalogo_ciudades = cities['city_ascii'].dropna().unique().tolist()
catalogo_ciudades

In [ ]:
# Funcion de homologar
def homologar_ciudad(ciudad):
    if pd.isna(ciudad) or ciudad.strip() == "":
        return "Other / Unknown"

    match = process.extractOne(
        ciudad,
        catalogo_ciudades,
        scorer=fuzz.token_sort_ratio
    )

    if match and match[1] >= 85:
        return match[0]
    else:
        return "Other / Unknown"

In [ ]:
df['city_std'] = df['city_clean'].apply(homologar_ciudad)

In [ ]:
# Como las ciudades son muchas se exporta para revisar
cities = df[df['city_std'] == 'Other / Unknown'][['City', 'city_clean', 'city_std']]
cities_df = pd.DataFrame(cities, columns=['City', 'city_clean', 'city_std'])
cities_df.to_csv("/content/drive/MyDrive/Visualizacion & Storytelling/semana3/cities5.csv", index=False, encoding="utf-8-sig")

In [ ]:
df[df['city_std'] == 'Other / Unknown'][['City', 'city_clean', 'city_std']]

In [ ]:
df.head(5)

### Limpiar y homologar industrias

In [ ]:
# Como las ciudades son muchas se exporta para revisar
import pandas as pd
indust = "/content/drive/MyDrive/Visualizacion & Storytelling//Trabajo_final_storytelling/Entrega2/Insumos/INDUSTRY.xlsx"
ind = pd.read_excel(indust)
ind



In [ ]:
# Ajustes a la tabla de homologacion manual

industry_red_fixes = {
    # Clear misclassifications in your INDUSTRY_RED
    'Fintech': 'Biotechnology',          # Biotechnology wrongly mapped to Fintech in your file
    'Coaching': 'Analytics',             # Data Analytics / Data Entry etc. mapped to Coaching
    'Public Library': 'Public Health',   # Public health mapped to Public Library
    'Editor': 'Education',               # Ed Tech mapped to Editor
}

industry_red_normalize = {
    # Pharma spelling
    'Pharmaceitical': 'Pharmaceutical',
    'Pharmaceitical ': 'Pharmaceutical',

    # Environmental spelling
    'Enviromental': 'Environmental',
    'Environmnetal': 'Environmental',

    # Standardize non-profit spellings
    'Non-profit': 'Nonprofit',
    'Non-profit ': 'Nonprofit',
    'Nonprofits': 'Nonprofit',

    # Standardize retail/wholesale casing bucket
    'wholesale': 'Wholesale',

    # Consistency for Life Sciences label variants (optional but helpful)
    'Life': 'Life Sciences',
}

# First: fix wrong mappings
ind['Industry_group'] = ind['INDUSTRY_RED'].replace(industry_red_fixes)

# Second: normalize spelling/variants
ind['Industry_group'] = ind['INDUSTRY_RED'].replace(industry_red_normalize)



In [ ]:
macro_map = {

# ================= TECHNOLOGY =================
'Technology': 'Technology',
'IT': 'Technology',
'Computing/Tech': 'Technology',
'Software': 'Technology',
'Saas': 'Technology',
'Internet': 'Technology',
'Telecommunications': 'Technology',
'digital': 'Technology',
'ECommerce': 'Technology',
'UX': 'Technology',
'User': 'Technology',

# ================= HEALTHCARE =================
'Life': 'Healthcare / Life Sciences',
'Biotechnology': 'Healthcare / Life Sciences',
'Biomedical': 'Healthcare / Life Sciences',
'Biologist': 'Healthcare / Life Sciences',
'biological': 'Healthcare / Life Sciences',
'Biology': 'Healthcare / Life Sciences',
'clinical': 'Healthcare / Life Sciences',
'Medical': 'Healthcare / Life Sciences',
'Pharmaceutical': 'Healthcare / Life Sciences',
'Pharmaceitical': 'Healthcare / Life Sciences',
'Diagnostic': 'Healthcare / Life Sciences',
'Preclinical': 'Healthcare / Life Sciences',
'Health': 'Healthcare / Life Sciences',
'Lab': 'Healthcare / Life Sciences',
'Toxicology': 'Healthcare / Life Sciences',
'Veterinary': 'Healthcare / Life Sciences',



# ================= EDUCATION =================
'Education': 'Education / Research',
'Research': 'Education / Research',
'University': 'Education / Research',
'School': 'Education / Research',
'ESL': 'Education / Research',
'student': 'Education / Research',
'Graduate': 'Education / Research',

# ================= GOVERNMENT =================
'Government': 'Government / Public Sector',
'State': 'Government / Public Sector',
'Federal': 'Government / Public Sector',
'Municipal': 'Government / Public Sector',
'Public Library': 'Government / Public Sector',

# ================= FINANCE =================
'Finance': 'Finance / Insurance',
'Insurance': 'Finance / Insurance',
'Fintech': 'Finance / Insurance',
'Banking': 'Finance / Insurance',
'Actuarial': 'Finance / Insurance',
'Mortgage': 'Finance / Insurance',
'Accounting, Banking & Finance': 'Finance / Insurance',

# ================= CONSULTING =================
'Consulting': 'Consulting / Professional Services',
'Consultant': 'Consulting / Professional Services',
'Management': 'Consulting / Professional Services',
'Professional': 'Consulting / Professional Services',
'Business or Consulting': 'Consulting / Professional Services',

# ================= RETAIL =================
'Retail': 'Retail / Consumer / Food',
'Consumer': 'Retail / Consumer / Food',
'CPG': 'Retail / Consumer / Food',
'Food': 'Retail / Consumer / Food',
'Restaurant': 'Retail / Consumer / Food',
'Restaurants': 'Retail / Consumer / Food',
'Grocery': 'Retail / Consumer / Food',

# ================= MANUFACTURING =================
'Manufacturing': 'Manufacturing / Industry',
'Industrial': 'Manufacturing / Industry',
'Production': 'Manufacturing / Industry',
'Warehouse': 'Manufacturing / Industry',
'Engineering or Manufacturing': 'Manufacturing / Industry',

# ================= ENERGY =================
'Energy': 'Energy / Utilities',
'Oil': 'Energy / Utilities',
'Gas': 'Energy / Utilities',
'Renewable': 'Energy / Utilities',
'Utilities': 'Energy / Utilities',

# ================= MEDIA =================
'Marketing': 'Media / Marketing / Publishing',
'Media & Digital': 'Media / Marketing / Publishing',
'Publishing': 'Media / Marketing / Publishing',
'Writing': 'Media / Marketing / Publishing',
'Journalism': 'Media / Marketing / Publishing',

# ================= NONPROFIT =================
'Nonprofit': 'Nonprofit / NGO',
'Non-profit': 'Nonprofit / NGO',
'Charity': 'Nonprofit / NGO',
'Philanthropy': 'Nonprofit / NGO',
'Nonprofits': 'Nonprofit / NGO',

# ================= LEGAL =================
'Law': 'Legal',
'Legal': 'Legal',
'Compliance': 'Legal',

# ================= REAL ESTATE =================
'Real': 'Real Estate / Construction',
'Construction': 'Real Estate / Construction',
'Property /Construction': 'Real Estate / Construction',
'Architecture': 'Real Estate / Construction',

# ================= TRANSPORT =================
'Transport': 'Transportation / Logistics',
'Logistics': 'Transportation / Logistics',
'Delivery': 'Transportation / Logistics',

# ================= HOSPITALITY =================
'Hospitality': 'Hospitality / Tourism',
'Travel': 'Hospitality / Tourism',
'Tourism': 'Hospitality / Tourism',



}
ind['Industry_group'] = ind['INDUSTRY_RED'].map(macro_map).fillna('Other')

In [ ]:
macro_map2 = {

# ============Cambiar other que no tienen sentido========
 'Nonprofits':  'Nonprofit / NGO',
 'Accounting, Banking & Finance': 'Finance / Insurance',
 'Engineering or Manufacturing': 'Manufacturing / Industry',
 'Business or Consulting': 'Consulting / Professional Services',
 'Social Work': 'Media / Marketing / Publishing',
 'Entertainment': 'Media / Marketing / Publishing',
 'Recruitment or HR': 'Recruitment / HR '

}

mask_other = ind["Industry_group"] == "Other"

ind.loc[mask_other, "Industry_group"] = (
    ind.loc[mask_other, "Industry"].map(macro_map2)
        .fillna("Other")
)

In [ ]:
ind['Industry_group'].value_counts()

In [ ]:
ind.loc[ind["Industry_group"] == "Other", "Industry"].value_counts().head(20)

In [ ]:
# Cruzar con base Industry homologado

df2 = df.merge(
    ind[["Industry", "Industry_group"]],
    on="Industry",
    how="left"
)

df2.head()

### Limpieza de salario y bono

In [ ]:
df2['Annual_salary'].unique()

In [ ]:
# Ajuste salario anual y adicional

def clean_money_strict(s: pd.Series) -> pd.Series:
    # string base
    x = s.astype(str).str.strip()

    # si tiene letras (a-z) => descartar
    has_letters = x.str.contains(r"[A-Za-z]", regex=True, na=False)

    # quitar símbolos y separadores comunes (espacios, $, comas, puntos)
    # ojo: esto asume que los separadores son solo formato, no decimales “reales” para salarios
    x = (x.str.replace(r"[\s\$,]", "", regex=True)
           .str.replace(".", "", regex=False)
           .str.replace(",", "", regex=False)
           .str.replace("+", "", regex=False))

    # normalizar vacíos
    x = x.replace({"": np.nan, "nan": np.nan, "None": np.nan, "-": np.nan})

    # convertir
    out = pd.to_numeric(x, errors="coerce")

    # aplicar regla: si tenía letras -> NaN
    out = out.mask(has_letters, np.nan)

    return out



In [ ]:
# 1) limpiar a numérico
df2["Annual_salary_n"] = clean_money_strict(df2["Annual_salary"])
df2["Aditional_salary_n"] = clean_money_strict(df2["Aditional_salary"])

MAX_VAL = 9_999_999

df2["Annual_salary_n"] = df2["Annual_salary_n"].where(
    df2["Annual_salary_n"] <= MAX_VAL
)

df2["Aditional_salary_n"] = df2["Aditional_salary_n"].where(
    df2["Aditional_salary_n"] <= MAX_VAL
)


In [ ]:
df2.columns

In [ ]:
# 2) si anual es 0 => adicional debe ser 0
df2.loc[df2["Annual_salary_n"].eq(0), "Aditional_salary_n"] = 0

# 3) adicional negativo => NaN (pero 0 se permite)
df2.loc[df2["Aditional_salary_n"] < 0, "Aditional_salary_n"] = np.nan

# 4) ajustar anual si tiene menos de 4 cifras (1 a 999) => *1000
mask_anual_baja = df2["Annual_salary_n"].between(1, 999, inclusive="both")
df2.loc[mask_anual_baja, "Annual_salary_n"] = df2.loc[mask_anual_baja, "Annual_salary_n"] * 1000

# 5) mínimo: todo lo NO-cero debe ser >= 1000; si no => NaN
MIN_SALARIO_ANUAL = 1000
df2.loc[df2["Annual_salary_n"].ne(0) & (df2["Annual_salary_n"] < MIN_SALARIO_ANUAL), "Annual_salary_n"] = np.nan

# Si hay salario anual (no NaN) y el adicional está NaN, asumir adicional = 0
df2.loc[df2["Annual_salary_n"].notna() & df2["Aditional_salary_n"].isna(), "Aditional_salary_n"] = 0
df2.loc[df2["Annual_salary_n"].eq(0), "Aditional_salary_n"] = 0



In [ ]:
print('Annual_salary_n:')
print('Min:' , df2['Annual_salary_n'].min())
print('Max' , df2['Annual_salary_n'].max())
print('Aditional_salary_n:')
print('Min: ', df2['Aditional_salary_n'].min())
print('Max: ', df2['Aditional_salary_n'].max())

#### Trtamiento de datos extremos - Outliers - Campos limpios **Annual_salary_n** y **Aditional_salary_n**

In [ ]:
cols = ["Annual_salary_n", "Aditional_salary_n"]

temp = df2.copy()

for c in cols:
    temp[c] = pd.to_numeric(temp[c], errors="coerce")

def get_iqr_bounds(series, k=1.5):
    Q1 = series.quantile(0.25)
    Q3 = series.quantile(0.75)
    IQR = Q3 - Q1
    lower = Q1 - k * IQR
    upper = Q3 + k * IQR
    return lower, upper

mask_keep = pd.Series(True, index=temp.index)

for c in cols:
    lower, upper = get_iqr_bounds(temp[c])

    mask_low  = temp[c] < lower
    mask_high = temp[c] > upper

    print(f"\nColumna: {c}")
    print(f"Los valores menores a {lower:,.2f} fueron eliminados → {mask_low.sum()} registros")
    print(f"Los valores mayores a {upper:,.2f} fueron eliminados → {mask_high.sum()} registros")

    mask_keep &= temp[c].between(lower, upper)

# 🔹 Aquí sí reemplazamos df2 con la versión limpia
df2 = df2.loc[mask_keep].copy()

print("\nResumen general")
print("Registros después de limpieza:", len(df2))


### Estandarización del tipo de monedas

In [ ]:
df2["Currency_std"] = (
    df2["Currency"]
      .astype(str)
      .str.strip()
      .str.upper()
)

# resolver el único caso especial
df2.loc[df2["Currency_std"] == "AUD/NZD", "Currency_std"] = "AUD"

df2["Other_currency_clean"] = (
    df2["Other_currency"]
      .astype(str)
      .str.strip()
      .str.upper()
)

df2["Currency_std"].unique()

df2["Other_currency_clean"].unique()

In [ ]:
map_other_currency = {

# USD
"USD": "USD",
"US DOLLAR": "USD",
"AMERICAN DOLLARS": "USD",

# ARS
"ARS": "ARS",
"ARGENTINE PESO": "ARS",
"PESO ARGENTINO": "ARS",
"ARGENTINIAN PESO (ARS)": "ARS",

# AUD
"AUD": "AUD",
"AUD AUSTRALIAN": "AUD",
"AUSTRALIAN DOLLARS": "AUD",

# BDT
"BDT": "BDT",

# BRL
"BRL": "BRL",
"BR$": "BRL",
"BRL (R$)": "BRL",

# CAD
"CAD": "CAD",

# CHF
"CHF": "CHF",

# CNY
"CNY": "CNY",
"CHINA RMB": "CNY",
"RMB (CHINESE YUAN)": "CNY",

# COP
"COP": "COP",
"PESOS COLOMBIANOS": "COP",

# HRK
"CROATIAN KUNA": "HRK",

# CZK
"CZK": "CZK",
"CZECH CROWNS": "CZK",

# DKK
"DKK": "DKK",
"DANISH KRONER": "DKK",

# EUR
"EUR": "EUR",
"EURO": "EUR",

# GBP
"GBP": "GBP",

# IDR
"IDR": "IDR",

# ILS
"ILS": "ILS",
"ILS (SHEKEL)": "ILS",
"ILS/NIS": "ILS",
"ISRAELI SHEKELS": "ILS",
"NIS (NEW ISRAELI SHEKEL)": "ILS",
"ILS ": "ILS",

# INR
"INR": "INR",
"INDIAN RUPEES": "INR",
"INR (INDIAN RUPEE)": "INR",
"RUPEES": "INR",

# KRW
"KRW": "KRW",
"KOREAN WON": "KRW",
"KRW (KOREAN WON)": "KRW",

# LKR
"LKR": "LKR",

# MXN
"MXN": "MXN",
"MEXICAN PESOS": "MXN",

# MYR
"MYR": "MYR",
"RM": "MYR",

# NGN
"NGN": "NGN",

# NOK
"NOK": "NOK",
"NORWEGIAN KRONER (NOK)": "NOK",

# TWD
"NTD": "TWD",
"TAIWANESE DOLLARS": "TWD",

# NZD
"NZD": "NZD",

# PHP
"PHP": "PHP",
"PHILIPPINE PESO": "PHP",
"PHILIPPINE PESO (PHP)": "PHP",
"PHILIPPINE PESOS": "PHP",
"PHP": "PHP",
"PHP (PHILIPPINE PESO)": "PHP",

# PLN
"PLN": "PLN",
"PLN (POLISH ZLOTY)": "PLN",
"PLN (ZWOTY)": "PLN",
"POLISH ZŁOTY": "PLN",

# CNY alias
"RMB": "CNY",

# SAR
"SAR": "SAR",

# SEK
"SEK": "SEK",

# SGD
"SGD": "SGD",
"SINGAPORE DOLLARA": "SGD",

# THB
"THB": "THB",
"THAI BAHT": "THB",
"THAI  BAHT": "THB",

# TRY
"TRY": "TRY",

# TTD
"TTD": "TTD",

# TZS
"TZS": "TZS",

# ZAR
"ZAR": "ZAR",
"ZAR ": "ZAR",

# ZMW
"ZMW": "ZMW",

# no moneda
"EQUITY": np.nan,
"(EN BLANCO)": np.nan,
}


In [ ]:
mask_other = df2["Currency_std"].eq("OTHER")

df2["Other_currency_std"] = np.nan
df2.loc[mask_other, "Other_currency_std"] = (
    df2.loc[mask_other, "Other_currency_clean"].map(map_other_currency)
)

df2["Currency_final"] = df2["Currency_std"]
df2.loc[mask_other, "Currency_final"] = df2.loc[mask_other, "Other_currency_std"]

In [ ]:
df2[["Currency", "Currency_std", "Other_currency", "Other_currency_std", "Currency_final"]].head(20)


In [ ]:
df2.loc[mask_other & df2["Other_currency_std"].isna(), "Other_currency_clean"].value_counts()


In [ ]:
df2["Currency_final"].unique()

### 5. Tasas

In [ ]:
fx_manual = {
    "COP": 1,

    "USD": 3684.35,
    "EUR": 4339.83,
    "GBP": 5012.26,
    "CAD": 2678.78,
    "AUD": 2583.84,
    "NZD": 2400.00,

    "CHF": 4724.26,
    "SEK": 380.00,
    "NOK": 350.00,
    "DKK": 580.00,

    "JPY": 23.60,
    "CNY": 510.00,
    "HKD": 467.99,
    "TWD": 115.00,
    "SGD": 2730.00,
    "KRW": 2.51,
    "THB": 105.00,
    "IDR": 0.24,
    "MYR": 780.00,
    "PHP": 66.00,

    "INR": 44.00,
    "LKR": 11.50,
    "BDT": 33.00,
    "SAR": 980.00,
    "ILS": 990.00,

    "ZAR": 228.87,
    "NGN": 4.50,
    "TZS": 1.50,
    "ZMW": 145.00,

    "ARS": 4.20,
    "MXN": 215.00,
    "BRL": 740.00,
    "TTD": 540.00,

    "CZK": 175.00,
    "PLN": 1000.00,
    "TRY": 115.00,
    "HRK": 575.00
}

In [ ]:
df2["FX_TO_COP"] = df2["Currency_final"].map(fx_manual)

In [ ]:
df2.head(3)

In [ ]:
df2[df2["FX_TO_COP"].isna()]["Currency_final"].value_counts()

In [ ]:
df2["FX_date"] = pd.Timestamp.now(tz="America/Bogota").date()

In [ ]:
df2.head(3)

In [ ]:
df2["Annual_salary_cop"] = df2["Annual_salary_n"] * df2["FX_TO_COP"]
df2["Aditional_salary_cop"] = df2["Aditional_salary_n"] * df2["FX_TO_COP"]

In [ ]:
pd.options.display.float_format = '{:,.0f}'.format

df2["Total_salary_cop"] = (
    df2["Annual_salary_cop"] + df2["Aditional_salary_cop"]
)

df2["Total_salary_mes_cop"] = (
    df2["Total_salary_cop"] / 12
)

cols = [
    "Annual_salary_cop",
    "Aditional_salary_cop",
    "Total_salary_cop",
    "Total_salary_mes_cop"
]

for c in cols:
    df2[c] = df2[c].round(0).astype("Int64")

In [ ]:
df2.head(3)

### Seleccion de campos finales y renombrar a Español

In [ ]:
df2.columns

In [ ]:
final_cols = ['Timestamp', 'Age', 'Industry', 'Job_title',  'Income_details',
              'Years_experience_overall',
              'Years_experience_field', 'Max_education_level', 'Gender', 'Race',
              'country_std', 'State',
              'city_std', 'ciudad_imputada_capital', 'Industry_group',
              'Annual_salary_n', 'Aditional_salary_n', 'Currency_final',
              'FX_TO_COP', 'FX_date', 'Annual_salary_cop', 'Aditional_salary_cop',
              'Total_salary_cop', 'Total_salary_mes_cop']

# Filtro estados unidos solamente

f = df2["country_std"] == 'United States'

df3 = df2.loc[f, final_cols].copy()

In [ ]:
rename_final = {
    'Timestamp': 'Fecha_registro',
    'Age': 'rango_edad',
    'Industry': 'industria',
    'Job_title': 'nombre_trabajo',
    'Income_details': 'info_adicionales_ingresos',
    'Years_experience_overall': 'experiencia_general',
    'Years_experience_field': 'experiencia_campo',
    'Max_education_level': 'nivel_educacion',
    'Gender': 'genero',
    'Race': 'etnia',
    'country_std': 'pais',
    'city_std': 'ciudad',
    'State': 'estado',
    'ciudad_imputada_capital': 'ciudad_capital',
    'Industry_group': 'industria_grupo',
    'Annual_salary_n': 'salario_anual',
    'Aditional_salary_n': 'salario_adicional_anual',
    'Currency_final': 'moneda',
    'FX_TO_COP': 'FX_TO_COP',
    'FX_date': 'FX_date',
    'Annual_salary_cop': 'salario_anual_cop',
    'Aditional_salary_cop': 'salario_adicional_anual_cop',
    'Total_salary_cop': 'salario_total_anual_cop',
    'Total_salary_mes_cop' : 'salario_mensual_total_cop'
}

In [ ]:
df3 = df3.rename(columns=rename_final)
df3.head(5)

### Ajuste de los estados - son necesarios para cruzar con el RPP y el GDP


In [ ]:
# Cuando trabaja en varios estados, dejar el primero
df3["estado"] = (
    df3["estado"]
    .str.split(",")
    .str[0]
    .str.strip()
)

#Eliminar los nan del estado

df3 = df3.dropna(subset=["estado"])

print(df3['estado'].unique())

### Crear campos de año y trimestre para el cruce con GDP y RPP

In [ ]:
df3["Fecha_registro"] = pd.to_datetime(df3["Fecha_registro"], errors="coerce")
df3["anio"] = df3["Fecha_registro"].dt.year
df3["anio"] = pd.to_numeric(df3["anio"], errors="coerce").astype("Int64")


In [ ]:
df3["Fecha_registro"] = pd.to_datetime(df3["Fecha_registro"])

In [ ]:
df3["anio_trimestre"] = (
    df3["Fecha_registro"].dt.year.astype(str)
    + ":Q"
    + df3["Fecha_registro"].dt.quarter.astype(str)
)

In [ ]:
df3[['Fecha_registro','anio', 'anio_trimestre', 'estado']].head(3)

### Regional Price Parities (RPP) by state - Indice de costo de vida por estado

Se utilizará el índice anual de Paridades Regionales de Precios publicado por el Bureau of Economic Analysis (BEA), que mide qué tan caro o barato es vivir en cada estado frente al promedio nacional de Estados Unidos (base 100). Este indicador tiene periodicidad anual y permite ajustar los salarios nominales para hacerlos comparables entre estados, teniendo en cuenta las diferencias en el costo de vida. El ajuste se realiza dividiendo el salario por (RPP/100), lo que permite estimar el salario en términos de poder adquisitivo real.

In [ ]:
# Cargar Datos
pathRPP = "/content/drive/MyDrive/Visualizacion & Storytelling/Trabajo_final_storytelling/Entrega2/Insumos/RPP_indice_precios_consumidor.csv"
RPP = pd.read_csv(
    pathRPP,
    skiprows=3  # ajusta si es necesario
    )

In [ ]:
RPP.head()

In [ ]:
# Tranformaciones

RPP = RPP[RPP["GeoName"] != "United States"].copy()
RPP["estado"] = RPP["GeoName"].str.strip().str.lower()
RPP.head(3)

In [ ]:
RPP_long = RPP.melt(
    id_vars=["GeoFIPS", "GeoName", "estado"],
    var_name="anio",
    value_name="rpp"
)

RPP_long["anio"] = pd.to_numeric(RPP_long["anio"], errors="coerce").astype("Int64")


In [ ]:
RPP_long.head(3)

In [ ]:
print(df3["anio"].dtype)
print(RPP_long["anio"].dtype)

In [ ]:
# Cruzar con base df3
df3["estado1"] = df3["estado"].str.strip().str.lower()
RPP_long["estado1"] = RPP_long["estado"].str.strip().str.lower()

df3 = df3.merge(
    RPP_long[["estado1", "anio", "rpp"]],
    on=["estado1", "anio"],
    how="left"
)

In [ ]:
# La base solamente se encuentra hasta 2024 por lo tanto 2025 y 2026 quedan vacios
f = df3['rpp'].isna()
pr = df3.loc[f, ["estado", "anio", "rpp"]]
pr['anio'].unique()

In [ ]:
# Salario real
pd.options.display.float_format = '{:.0f}'.format
df3["salario_mensual_real_ajustado"] = df3["salario_mensual_total_cop"] / (df3["rpp"] / 100)
df3["salario_anual_real_ajustado"] = df3["salario_total_anual_cop"] / (df3["rpp"] / 100)

In [ ]:
df3.head(3)

In [ ]:
df3.columns

###  Gross Domestic Product (GDP) by State - PIB real trimestral por estado - Millones de dolares.

Se utilizará el Producto Interno Bruto real trimestral por estado, también publicado por el BEA dentro de las Cuentas Económicas Regionales. Este indicador mide el nivel de producción económica de cada estado, ajustado por inflación, y tiene periodicidad trimestral. En el modelo se empleará como variable de contexto económico para representar el nivel de actividad económica del estado y analizar su relación con el salario real.

In [ ]:
# Cargar Datos
pathGDP = "/content/drive/MyDrive/Visualizacion & Storytelling/Trabajo_final_storytelling/Entrega2/Insumos/GDP_bienes_y_servicios_producidos_por_estado_trimestral.csv"
GDP = pd.read_csv(
    pathGDP,
    skiprows=3  # ajusta si es necesario
    )

In [ ]:
GDP.head(5)

In [ ]:
# Transformaciones
GDP = GDP[GDP["GeoName"] != "United States"].copy()
GDP["estado1"] = GDP["GeoName"].str.strip().str.lower()

In [ ]:
# Despivotar

columnas_periodo = [c for c in GDP.columns if ":" in c]

GDP_long = GDP.melt(
    id_vars=["estado1"],
    value_vars=columnas_periodo,
    var_name="anio_trimestre",
    value_name="gdp"
)

GDP_long["gdp"] = pd.to_numeric(GDP_long["gdp"], errors="coerce")



In [ ]:
GDP_long.head()

In [ ]:
# Cruzar con base df3
df3 = df3.merge(
    GDP_long[["estado1", "anio_trimestre", "gdp"]],
    on=["estado1", "anio_trimestre"],
    how="left"
)

In [ ]:
# La tabla de GDP tiene solamente hasta 2025:Q3 por eso 2025:Q4 y 2026:Q1 quedan sin asignar
f = df3['gdp'].isna()
pr = df3.loc[f, ["estado", "anio_trimestre", "gdp"]]
# pr.head(10)
pr['anio_trimestre'].unique()

In [ ]:
# Crear GDP pesos COP
df3["gdp_cop"] = df3["gdp"] * 1_000_000 * df3["FX_TO_COP"]
df3.head(5)

In [ ]:
df3.columns

In [ ]:
# Borrar cols Innecesarias
cols_elimi = ['rpp_x','estado1','rpp_y', 'info_adicionales_ingresos']
df3 = df3.drop(columns=cols_elimi, errors="ignore")

In [ ]:
df3.columns

In [ ]:
df3.head(3)

### Guardado base final

In [ ]:
pathSAL = "/content/drive/MyDrive/Visualizacion & Storytelling/Trabajo_final_storytelling/Entrega2/Salidas/SALARY_SALIDA_FINAL.csv"
df3.to_csv(pathSAL, index=False)



In [ ]:
pathSALex = "/content/drive/MyDrive/Visualizacion & Storytelling/Trabajo_final_storytelling/Entrega2/Salidas/SALARY_SALIDA_FINAL.xlsx"
df3.to_excel(pathSALex, index=False)